# 06 — Pemët e vendimit dhe metodat ansambël

        **Pyetja udhëheqëse:** Si ndërtohet një vendim me rregulla dhe si stabilizohet me shumë pemë?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të kuptohen Gini, entropia dhe fitimi i informacionit.
- Të analizohet thellësia dhe mbipërshtatja.
- Të krahasohet një pemë me Random Forest.
- Të interpretohen rëndësitë e veçorive.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

## Pse kjo temë është e nevojshme?

Pemët ndërtojnë një vendim përmes pyetjeve të njëpasnjëshme mbi veçoritë. Kjo i bën tërheqëse për interpretim, por edhe të ndjeshme ndaj të dhënave. Ansamblet si Random Forest shfrytëzojnë shumë modele të paqëndrueshme për të prodhuar një parashikim më të qëndrueshëm.

Do të lidhim papastërtinë, fitimin e informacionit, thellësinë dhe bootstrap-in me idenë statistikore të uljes së variancës. Rëndësia e veçorive do të trajtohet si mjet diagnostik, jo si provë e shkakësisë.

### Si duhet lexuar ky notebook

Për çdo bllok kodi do të shpjegojmë **pse** kryhet ai hap, çfarë pyetjeje i përgjigjet dhe çfarë duhet të kontrollojmë në dalje. Kodi nuk trajtohet si recetë: çdo grafik dhe çdo metrikë duhet të rikthehet te hipoteza matematikore dhe te interpretimi fizik. Kur një rezultat nuk sillet siç presim, pyetja e parë nuk është “si ta rrisim score-in?”, por “cilën hipotezë të modelit ose të të dhënave po shkelim?”.

### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("06 — Pemët e vendimit dhe metodat ansambël")
print("Seed:", RANDOM_STATE)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 1. Klasifikimi i fazave të një materiali — CORE

### Para se të ekzekutojmë: Një diagram fazash si problem klasifikimi

Simulojmë një kufi fazor me zhurmë në etiketa. Pemët janë të përshtatshme për këtë shembull sepse ndajnë hapësirën me rregulla të tipit “nëse $T$ është mbi këtë vlerë…”.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Një diagram fazash si problem klasifikimi.
n=600
T=rng.uniform(100,900,n)
P=rng.uniform(0,10,n)
phase=((T>500+18*(P-5)) | ((T>420)&(P>7))).astype(int)
flip=rng.random(n)<0.07
phase[flip]=1-phase[flip]
X=np.c_[T,P]; y=phase
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(T,P,c=y,s=12)
ax.set_xlabel("Temperatura"); ax.set_ylabel("Presioni")
ax.set_title("Faza materiale në hapësirën (T,P)")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 2. Gini dhe entropia — CORE

Për probabilitete klase $p_k$,
$$
G=1-\sum_k p_k^2,\qquad
H=-\sum_k p_k\log_2 p_k.
$$
Një ndarje e mirë redukton papastërtinë e nyjeve pasardhëse.

### Para se të ekzekutojmë: Papastërtia si funksion i përzierjes së klasave

Vizualizojmë Gini dhe entropinë për një problem binar. Të dyja janë zero kur nyja është e pastër dhe maksimale kur klasat janë të përziera. Këto madhësi udhëheqin zgjedhjen e ndarjeve të pemës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Papastërtia si funksion i përzierjes së klasave.
p=np.linspace(0.001,0.999,400)
g=1-(p**2+(1-p)**2)
h=-(p*np.log2(p)+(1-p)*np.log2(1-p))
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(p,g,label="Gini")
ax.plot(p,h,label="Entropia")
ax.set_xlabel("p"); ax.set_ylabel("Papastërtia")
ax.legend(); ax.set_title("Dy masa të papastërtisë")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 3. Një pemë e vetme — CORE

### Para se të ekzekutojmë: Struktura e një peme konkrete

Trajnojmë një pemë të cekët dhe e vizualizojmë të plotë. Kjo na lejon të lexojmë rregullat e vendimit dhe të lidhim çdo ndarje me një prag në një madhësi fizike.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Struktura e një peme konkrete.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.tree import DecisionTreeClassifier, plot_tree
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.model_selection import train_test_split
# Ndajmë të dhënat në mënyrë të riprodhueshme; test-i nuk përdoret për trajnim.
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,stratify=y,random_state=RANDOM_STATE)
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
tree=DecisionTreeClassifier(max_depth=3,random_state=RANDOM_STATE).fit(Xtr,ytr)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots(figsize=(12,6))
plot_tree(tree,feature_names=["T","P"],class_names=["faza0","faza1"],filled=True,ax=ax)
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

### Para se të ekzekutojmë: Struktura e një peme konkrete

Trajnojmë një pemë të cekët dhe e vizualizojmë të plotë. Kjo na lejon të lexojmë rregullat e vendimit dhe të lidhim çdo ndarje me një prag në një madhësi fizike.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Struktura e një peme konkrete.
depths=range(1,16)
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
tr=[DecisionTreeClassifier(max_depth=d,random_state=RANDOM_STATE).fit(Xtr,ytr).score(Xtr,ytr) for d in depths]
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
te=[DecisionTreeClassifier(max_depth=d,random_state=RANDOM_STATE).fit(Xtr,ytr).score(Xte,yte) for d in depths]
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(list(depths),tr,marker="o",label="trajnim")
ax.plot(list(depths),te,marker="o",label="provë")
ax.set_xlabel("Thellësia"); ax.set_ylabel("Accuracy"); ax.legend()
ax.set_title("Mbipërshtatja e pemës")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 4. Random Forest: bagging dhe rastësi në veçori — CORE

### Para se të ekzekutojmë: Nga një pemë te një ansambël

Random Forest ndërton shumë pemë mbi mostra bootstrap dhe zgjedhje të rastësishme veçorish. Mesatarizimi ul variancën dhe e bën parashikimin më të qëndrueshëm se një pemë e vetme.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Nga një pemë te një ansambël.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.ensemble import RandomForestClassifier
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
rf=RandomForestClassifier(n_estimators=250,random_state=RANDOM_STATE).fit(Xtr,ytr)
print("Tree:",tree.score(Xte,yte),"RF:",rf.score(Xte,yte))

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

### Para se të ekzekutojmë: Rëndësia e veçorive nga pemët

Rëndësia e bazuar në reduktimin e papastërtisë është e shpejtë, por mund të jetë e anshme. E përdorim si një pikënisje, jo si përfundim shkencor.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Rëndësia e veçorive nga pemët.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.bar(["T","P"],rf.feature_importances_)
ax.set_ylabel("Rëndësia")
ax.set_title("Rëndësia e veçorive në Random Forest")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 5. Permutation importance — CORE

### Para se të ekzekutojmë: Rëndësia me perturbim të kontrolluar

Permutation importance pyet sa bie performanca kur një veçori prishet duke u permutuar. Kjo e lidh rëndësinë me performancën e modelit në të dhëna të mbajtura mënjanë.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Rëndësia me perturbim të kontrolluar.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.inspection import permutation_importance
# Perturbohet secila veçori për të matur sa varet performanca prej saj.
perm=permutation_importance(rf,Xte,yte,n_repeats=30,random_state=RANDOM_STATE)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.bar(["T","P"],perm.importances_mean,yerr=perm.importances_std)
ax.set_ylabel("Rënie e score pas permutimit")
ax.set_title("Permutation importance")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Boosting: ideja — EXTENDED

Në boosting, modelet ndërtohen sekuencialisht dhe secili përpiqet të korrigjojë gabimet e ansamblit
të deritanishëm. Kjo është ndryshe nga bagging, ku modelet mund të ndërtohen paralelisht mbi mostra
bootstrap.

### Para se të ekzekutojmë: Ideja e boosting

Ndryshe nga bagging, boosting ndërton modele në sekuencë. Çdo model i ri përpiqet të korrigjojë atë që ansambli aktual nuk shpjegon mirë. Ky është vetëm një hyrje konceptuale, jo trajtim i plotë i boosting-ut.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Ideja e boosting.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.ensemble import GradientBoostingClassifier
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
gb=GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr,ytr)
print("Gradient Boosting test accuracy:",gb.score(Xte,yte))

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 7. Aplikim jo-fizik: vlerësim rreziku — EXTENDED

Pemët janë tërheqëse në fusha ku rregullat e vendimit duhet të shpjegohen, si sigurimet ose triage mjekësor. Megjithatë, interpretueshmëria e një peme nuk transferohet automatikisht te një pyll me qindra pemë.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Llogaritni Gini për një nyje me raport klasash 70:30.
2. Krahasoni `max_depth=2` dhe `max_depth=None`.
3. Shtoni një veçori zhurmë dhe analizoni rëndësinë e saj.
4. Krahasoni impurity-based dhe permutation importance.

        ## B. Ushtrim i plotë
        Ndërtoni një klasifikues fazash me 3 klasa dhe analizoni kufijtë e vendimit të një peme dhe Random Forest.

        ## C. Playground
        Ndryshoni thellësinë, numrin e pemëve, zhurmën e etiketave dhe seed.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Pemët e vendimit dhe metodat ansambël** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.